In [1]:
!pip -q install pyspark
from pyspark.sql import SparkSession, functions as F
import os, time
spark = SparkSession.builder.appName("RetailMartArchitecture").getOrCreate()
CLEAN_PATH = "/content/retailmart_output/retailmart_transactions_cleaned"
PARQUET_PATH = "/content/retailmart_output/retailmart_transactions_cleaned_parquet"

In [3]:
!pip install -q findspark
import os
import findspark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
findspark.init()
from pyspark.sql import SparkSession
# Create Spark Session with Hive support enabled
spark = SparkSession.builder \
 .master("local[*]") \
 .appName("Practical2_Hive_Spark") \
 .config("spark.sql.warehouse.dir", "/content/spark-warehouse") \
 .enableHiveSupport() \
 .getOrCreate()
print("✅ Spark session with Hive support created!")
print(f"Spark version: {spark.version}")

✅ Spark session with Hive support created!
Spark version: 4.0.4


In [6]:
from google.colab import files
uploaded = files.upload()

# Automatically detect the uploaded file's name
uploaded_file_name = list(uploaded.keys())[0]
print(f"Detected uploaded file: {uploaded_file_name}")

# Load the uploaded CSV using spark
df = spark.read.option("header", True).option("inferSchema", True).csv(uploaded_file_name)
print(f"\u2705 Dataset loaded: {df.count():,} rows, {len(df.columns)} columns")
df.show(5)

Saving retailmart_transactions_cleaned.csv to retailmart_transactions_cleaned (1).csv
Detected uploaded file: retailmart_transactions_cleaned (1).csv
✅ Dataset loaded: 990,078 rows, 16 columns
+--------------+--------------------+-------------------+----------------+---------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+-------------------+
|transaction_id|         customer_id|   transaction_date|product_category|   product_name|quantity|unit_price|total_amount|store_location|payment_method|customer_age|customer_gender|loyalty_years|num_visits|churn|     transaction_ts|
+--------------+--------------------+-------------------+----------------+---------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+-------------------+
|  DUP-00000002|e5fd51bb-e050-43f...|2022-02-09 08:24:36|          Beauty|     Makeup Kit|     1.0|     2

In [10]:
# Create a temporary view
df.createOrReplaceTempView("transactions")
# Query using Spark SQL on the actual loaded dataset
print("📊 Sample of aggregated transactional data using SQL:")
spark.sql("""
SELECT store_location,
       COUNT(*) as transaction_count,
       ROUND(AVG(total_amount), 2) as avg_transaction_amount
FROM transactions
GROUP BY store_location
ORDER BY transaction_count DESC
LIMIT 10
""").show()

📊 Sample of aggregated transactional data using SQL:
+----------------+-----------------+----------------------+
|  store_location|transaction_count|avg_transaction_amount|
+----------------+-----------------+----------------------+
|     Chicago, IL|            49623|                172.38|
|      Dallas, TX|            49472|                174.02|
|      Boston, MA|            49432|                172.61|
|    New York, NY|            49431|                 172.1|
| Minneapolis, MN|            49404|                172.09|
| Los Angeles, CA|            49387|                168.44|
|     Seattle, WA|            49376|                173.23|
|   Nashville, TN|            49372|                170.73|
|Philadelphia, PA|            49330|                171.98|
|    Portland, OR|            49312|                170.87|
+----------------+-----------------+----------------------+



In [8]:
# Create a managed table
spark.sql("CREATE DATABASE IF NOT EXISTS telco_db")
spark.sql("USE telco_db")
# Create table using SQL
df.write.mode("overwrite").saveAsTable("customers_table")
print("✅ Table created: telco_db.customers_table")
# Verify
spark.sql("SHOW TABLES").show()
spark.sql("SELECT COUNT(*) FROM customers_table").show()

✅ Table created: telco_db.customers_table
+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
| telco_db|customers_table|      false|
|         |   transactions|       true|
+---------+---------------+-----------+

+--------+
|count(1)|
+--------+
|  990078|
+--------+



In [11]:
from pyspark.sql.functions import year, month, date_format

# Add partition column using the correct 'transaction_date' from your RetailMart dataset
df_partitioned = df.withColumn(
    "year_month",
    date_format("transaction_date", "yyyy-MM")
)

# Show partition distribution
print("📊 Partition distribution:")
df_partitioned.groupBy("year_month").count().orderBy("year_month").show(20)

# Create partitioned table
df_partitioned.write.mode("overwrite") \
    .partitionBy("year_month") \
    .saveAsTable("transactions_partitioned")
print("✅ Partitioned table created: telco_db.transactions_partitioned")

# Query with partition pruning using a completed multi-line query string
print("📊 Query using partition pruning:")
spark.sql("""
SELECT COUNT(*)
FROM transactions_partitioned
WHERE year_month = '2025-01'
""").show()

📊 Partition distribution:
+----------+-----+
|year_month|count|
+----------+-----+
|   2021-01|15192|
|   2021-02|13696|
|   2021-03|15141|
|   2021-04|14419|
|   2021-05|15161|
|   2021-06|14957|
|   2021-07|15181|
|   2021-08|14971|
|   2021-09|14469|
|   2021-10|15086|
|   2021-11|14474|
|   2021-12|15096|
|   2022-01|15104|
|   2022-02|13496|
|   2022-03|15266|
|   2022-04|14522|
|   2022-05|14782|
|   2022-06|14550|
|   2022-07|15117|
|   2022-08|15148|
+----------+-----+
only showing top 20 rows
✅ Partitioned table created: telco_db.transactions_partitioned
📊 Query using partition pruning:
+--------+
|count(1)|
+--------+
|   14977|
+--------+



In [13]:
# 1. Total number of transactions and age statistics by store location
print("📊 1. Transactions by Store Location:")
spark.sql("""
 SELECT store_location,
        COUNT(*) as total_transactions,
        ROUND(AVG(total_amount), 2) as avg_transaction_amount,
        ROUND(AVG(customer_age), 1) as avg_customer_age
 FROM customers_table
 GROUP BY store_location
 ORDER BY total_transactions DESC
 LIMIT 10
""").show()

# 2. Churn analysis on retail transactions
print("📊 2. Churn rate by product category:")
spark.sql("""
 SELECT product_category,
        COUNT(*) as total,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) as churned_customers,
        ROUND(SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate
 FROM customers_table
 GROUP BY product_category
 ORDER BY churn_rate DESC
""").show()

# 3. Revenue analysis by payment method
print("📊 3. Revenue generated by Payment Method:")
spark.sql("""
 SELECT payment_method,
        ROUND(AVG(total_amount), 2) as avg_amount,
        ROUND(SUM(total_amount), 2) as total_revenue,
        COUNT(*) as transaction_count
 FROM customers_table
 GROUP BY payment_method
 ORDER BY total_revenue DESC
""").show()

📊 1. Transactions by Store Location:
+----------------+------------------+----------------------+----------------+
|  store_location|total_transactions|avg_transaction_amount|avg_customer_age|
+----------------+------------------+----------------------+----------------+
|     Chicago, IL|             49623|                172.38|            41.8|
|      Dallas, TX|             49472|                174.02|            41.9|
|      Boston, MA|             49432|                172.61|            41.8|
|    New York, NY|             49431|                 172.1|            41.9|
| Minneapolis, MN|             49404|                172.09|            41.9|
| Los Angeles, CA|             49387|                168.44|            41.9|
|     Seattle, WA|             49376|                173.23|            41.8|
|   Nashville, TN|             49372|                170.73|            42.0|
|Philadelphia, PA|             49330|                171.98|            41.9|
|    Portland, OR|         

In [18]:
# 4. Customer segmentation using actual transactional features
print("📊 4. Customer segments based on loyalty years and transactions:")
spark.sql("""
 SELECT
     CASE
         WHEN loyalty_years < 2 THEN 'New'
         WHEN loyalty_years < 5 THEN 'Regular'
         WHEN loyalty_years < 8 THEN 'Loyal'
         ELSE 'VIP'
     END as segment,
     COUNT(*) as customers,
     ROUND(AVG(total_amount), 2) as avg_transaction_amount,
     ROUND(AVG(num_visits), 2) as avg_visits,
     ROUND(SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate
 FROM customers_table
 GROUP BY
     CASE
         WHEN loyalty_years < 2 THEN 'New'
         WHEN loyalty_years < 5 THEN 'Regular'
         WHEN loyalty_years < 8 THEN 'Loyal'
         ELSE 'VIP'
     END
 ORDER BY churn_rate DESC
""").show()

📊 4. Customer segments based on loyalty years and transactions:
+-------+---------+----------------------+----------+----------+
|segment|customers|avg_transaction_amount|avg_visits|churn_rate|
+-------+---------+----------------------+----------+----------+
|    New|   481298|                 171.9|      5.84|     30.61|
|  Loyal|   118361|                171.97|     16.56|     14.62|
|Regular|   321441|                172.82|     10.55|     14.50|
|    VIP|    68978|                172.66|     23.44|     14.43|
+-------+---------+----------------------+----------+----------+



In [22]:
import time
from pyspark.sql.functions import date_format

print("=" * 70)
print("⏱️ PERFORMANCE COMPARISON")
print("=" * 70)

# Safely verify/add column without self-overwrite on Hive table
df_base = spark.table("customers_table")
if "year_month" not in df_base.columns:
    df_base = df_base.withColumn("year_month", date_format("transaction_date", "yyyy-MM"))
df_base.createOrReplaceTempView("customers_view")

# 1. Query on non-partitioned table/view
print("\n📊 1. Non-partitioned table query:")
start = time.time()
result1 = spark.sql("SELECT * FROM customers_view WHERE year_month = '2025-01'")
result1_count = result1.count()
end = time.time()
print(f" Count: {result1_count:,} rows")
print(f" Time: {end - start:.2f} seconds")

# 2. Query on partitioned table
print("\n📊 2. Partitioned table query:")
start = time.time()
result2 = spark.sql("SELECT * FROM transactions_partitioned WHERE year_month = '2025-01'")
result2_count = result2.count()
end = time.time()
print(f" Count: {result2_count:,} rows")
print(f" Time: {end - start:.2f} seconds")

# 3. Query on cached DataFrame
print("\n📊 3. Query on cached DataFrame:")
start = time.time()
df_partitioned.cache()
result3 = df_partitioned.filter("year_month = '2025-01'")
result3_count = result3.count()
end = time.time()
print(f" Count: {result3_count:,} rows")
print(f" Time: {end - start:.2f} seconds")

print("\n✅ Performance comparison complete!")

⏱️ PERFORMANCE COMPARISON

📊 1. Non-partitioned table query:
 Count: 14,977 rows
 Time: 1.11 seconds

📊 2. Partitioned table query:
 Count: 14,977 rows
 Time: 0.23 seconds

📊 3. Query on cached DataFrame:
 Count: 14,977 rows
 Time: 22.89 seconds

✅ Performance comparison complete!


In [23]:
print("=" * 70)
print("📋 STORAGE FORMAT COMPARISON")
print("=" * 70)
# 1. CSV format (text)
print("\n📊 1. CSV (text) format:")
df.write.mode("overwrite").option("header", True).csv("temp_csv")
!du -sh temp_csv
# 2. Parquet format
print("\n📊 2. Parquet (columnar) format:")
df.write.mode("overwrite").parquet("temp_parquet")
!du -sh temp_parquet
# 3. ORC format
print("\n📊 3. ORC (columnar) format:")
df.write.mode("overwrite").orc("temp_orc")
!du -sh temp_orc


📋 STORAGE FORMAT COMPARISON

📊 1. CSV (text) format:
175M	temp_csv

📊 2. Parquet (columnar) format:
69M	temp_parquet

📊 3. ORC (columnar) format:
42M	temp_orc


In [15]:
# We will save our loaded dataset to Parquet and CSV directories first to perform the benchmarking comparisons
import os
os.makedirs("/content/retailmart_output", exist_ok=True)

# Save as standard multi-part CSV
df.write.mode("overwrite").option("header", True).csv(CLEAN_PATH)

# Save as Parquet
df.write.mode("overwrite").parquet(PARQUET_PATH)

print("✅ Saved CSV to:", CLEAN_PATH)
print("✅ Saved Parquet to:", PARQUET_PATH)

# Re-read to confirm
df_csv = spark.read.option("header", True).option("inferSchema", True).csv(CLEAN_PATH)
print(f"Successfully re-read CSV: {df_csv.count():,} rows")

✅ Saved CSV to: /content/retailmart_output/retailmart_transactions_cleaned
✅ Saved Parquet to: /content/retailmart_output/retailmart_transactions_cleaned_parquet
Successfully re-read CSV: 990,078 rows


In [20]:
# Compare storage size (directory sizes)
def dir_size_mb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return round(total/(1024*1024), 2)
print("📊 CSV directory size:", dir_size_mb(CLEAN_PATH), "MB")
print("📊 Parquet directory size:", dir_size_mb(PARQUET_PATH), "MB")

📊 CSV directory size: 174.25 MB
📊 Parquet directory size: 68.37 MB


In [21]:
# Demonstrate distributed transformations and predicate filtering on the Spark DataFrame
start = time.time()
category_counts = df.groupBy("product_category").agg(
    F.count("*").alias("transactions"),
    F.round(F.sum("total_amount"), 2).alias("revenue")
).orderBy(F.desc("revenue"))
category_counts.show(truncate=False)
print("⏱️ Group-by elapsed seconds:", round(time.time()-start, 2))

start = time.time()
filter_count = df.filter(F.col("transaction_date") >= F.lit("2025-01-01")).select("transaction_id","transaction_date","total_amount").count()
print(f"⏱️ Filtered count ({filter_count:,} rows) elapsed seconds:", round(time.time()-start, 2))

+----------------+------------+-------------+
|product_category|transactions|revenue      |
+----------------+------------+-------------+
|Electronics     |124163      |7.79243149E7 |
|Home & Kitchen  |123514      |2.652941999E7|
|Sports          |124112      |2.352033359E7|
|Clothing        |123572      |1.411169892E7|
|Toys            |124188      |1.104890737E7|
|Beauty          |123228      |7876655.7    |
|Books           |123451      |5671620.65   |
|Groceries       |123850      |3868330.04   |
+----------------+------------+-------------+

⏱️ Group-by elapsed seconds: 4.16
⏱️ Filtered count (280,137 rows) elapsed seconds: 3.95
